# TDI Python 2.0 | Week 3: Data Structures & the Supply Chain Network
**Project:** Supply Chain Network and Single-Source Risk Report  
**Tools:** Python standard library (`csv`), lists, dictionaries, sets and tuples  

## Load dataset and define the cleaning helpers
Supplier names appear with inconsistent spacing and capitalization. I used the supplied Week 3 helper functions to standardize supplier names and trim category whitespace. Delivery statuses were inspected separately rather than silently corrected.

In [2]:
import csv

with open("supply_chain_dataset.csv", newline="", encoding="utf-8-sig") as file:
    rows = list(csv.DictReader(file))

def clean_supplier(name):
    """'Supplier A', 'suppliera', 'SupplierA' -> 'SupplierA'."""
    compact = name.strip().lower().replace(" ", "")
    return "Supplier" + compact[-1].upper()

def clean_category(category):
    """' Motors ' -> 'Motors'."""
    return category.strip()

print(f"Loaded {len(rows):,} orders")
print("Columns:", list(rows[0]))

Loaded 3,000 orders
Columns: ['order_id', 'order_date', 'product_category', 'quantity', 'unit_price', 'total_cost', 'supplier_name', 'supplier_location', 'lead_time_days', 'warehouse_destination', 'delivery_date', 'actual_delivery_days', 'delivery_status', 'on_time', 'product_defects', 'return_status']


## Task 1: Organise orders in a list
I converted valid `total_cost` values to numbers, then calculated total spending, average, median, and the five smallest and largest costs. Empty values were excluded from numerical calculations.

In [3]:
order_costs = [float(row["total_cost"]) for row in rows if row["total_cost"].strip()]
sorted_costs = sorted(order_costs)
count = len(sorted_costs)
midpoint = count // 2
median_cost = (sorted_costs[midpoint] if count % 2 else
               (sorted_costs[midpoint - 1] + sorted_costs[midpoint]) / 2)
print(f"Orders with recorded costs: {count:,}")
print(f"Total spend: {sum(order_costs):,.2f}")
print(f"Average order value: {sum(order_costs)/count:,.2f}")
print(f"Median order value: {median_cost:,.2f}")
print("Five smallest:", [round(x, 2) for x in sorted_costs[:5]])
print("Five largest:", [round(x, 2) for x in sorted_costs[-5:]])

Orders with recorded costs: 3,000
Total spend: 170,038,100.28
Average order value: 56,679.37
Median order value: 30,911.19
Five smallest: [-207.32, 335.27, 340.35, 340.52, 352.6]
Five largest: [765820.16, 807800.07, 991898.31, 1000681.24, 1720595.4]


## Task 2: Map product categories to supplier sets
I created a dictionary that maps each cleaned category to a **set** of cleaned supplier names. I also used a set to keep all distinct suppliers without duplicates.

In [4]:
category_suppliers = {}
for row in rows:
    category = clean_category(row["product_category"])
    supplier = clean_supplier(row["supplier_name"])
    category_suppliers.setdefault(category, set()).add(supplier)
for category in sorted(category_suppliers):
    print(f"{category}: {', '.join(sorted(category_suppliers[category]))}")
print("\nPumps suppliers:", sorted(category_suppliers.get("Pumps", set())))
all_suppliers = set().union(*category_suppliers.values())
print("Unique cleaned suppliers:", len(all_suppliers))

Connectors: SupplierE
Control Systems: SupplierA
Fasteners: SupplierE
Hydraulics: SupplierD
Motors: SupplierA, SupplierB
Pumps: SupplierC, SupplierD
Sensors: SupplierB, SupplierC
Valves: SupplierB, SupplierC

Pumps suppliers: ['SupplierC', 'SupplierD']
Unique cleaned suppliers: 5


## Task 3: Find unique values with sets
I compared the raw and cleaned names/categories. Status labels were compared using basic whitespace/case normalization **only for diagnosis**, and possible typos were displayed for review. I then demonstrated intersection, union and difference.

In [5]:
raw_suppliers = {row["supplier_name"] for row in rows}
clean_suppliers = {clean_supplier(row["supplier_name"]) for row in rows}
raw_categories = {row["product_category"] for row in rows}
clean_categories = {clean_category(row["product_category"]) for row in rows}
raw_statuses = {row["delivery_status"] for row in rows}
normalized_statuses = {s.strip().lower() for s in raw_statuses}
for label, before, after in [
    ("Supplier names", raw_suppliers, clean_suppliers),
    ("Categories", raw_categories, clean_categories),
    ("Delivery statuses", raw_statuses, normalized_statuses)]:
    print(f"{label}: {len(before)} raw, {len(after)} normalized")
print("Raw statuses (review possible typos):", sorted(raw_statuses))
print("Normalized status labels:", sorted(normalized_statuses))
motors = category_suppliers.get("Motors", set())
pumps = category_suppliers.get("Pumps", set())
print("Motors AND Pumps:", sorted(motors & pumps))
print("Motors OR Pumps:", sorted(motors | pumps))
print("Motors NOT Pumps:", sorted(motors - pumps))

Supplier names: 15 raw, 5 normalized
Categories: 8 raw, 8 normalized
Delivery statuses: 6 raw, 6 normalized
Raw statuses (review possible typos): ['Cancelled', 'Delayed', 'Delievered', 'Delivered', 'In-Transit', 'Pendng']
Normalized status labels: ['cancelled', 'delayed', 'delievered', 'delivered', 'in-transit', 'pendng']
Motors AND Pumps: []
Motors OR Pumps: ['SupplierA', 'SupplierB', 'SupplierC', 'SupplierD']
Motors NOT Pumps: ['SupplierA', 'SupplierB']


## Task 4: Single-source risk report
This report shows that any category supplied by only one supplier is vulnerable to disruption if that supplier becomes unavailable.

In [6]:
single_source = {category: next(iter(suppliers))
                 for category, suppliers in category_suppliers.items()
                 if len(suppliers) == 1}
print("=== SINGLE-SOURCE SUPPLY RISK REPORT ===")
if single_source:
    for category, supplier in sorted(single_source.items()):
        print(f"RISK | {category}: {supplier} is the only supplier")
else:
    print("No single-source categories found")
print(f"{len(single_source)} of {len(category_suppliers)} categories are single-source")

=== SINGLE-SOURCE SUPPLY RISK REPORT ===
RISK | Connectors: SupplierE is the only supplier
RISK | Control Systems: SupplierA is the only supplier
RISK | Fasteners: SupplierE is the only supplier
RISK | Hydraulics: SupplierD is the only supplier
4 of 8 categories are single-source


## Task 5: Build the supplier network
I created a nested dictionary to store the category set, order count and total recorded spend for each cleaned supplier. Missing costs were excluded from spend, but the corresponding orders still counted.

In [7]:
network = {}
for row in rows:
    supplier = clean_supplier(row["supplier_name"])
    category = clean_category(row["product_category"])
    if supplier not in network:
        network[supplier] = {"categories": set(), "order_count": 0, "total_spend": 0.0}
    network[supplier]["categories"].add(category)
    network[supplier]["order_count"] += 1
    if row["total_cost"].strip():
        network[supplier]["total_spend"] += float(row["total_cost"])
print("=== SUPPLIER A DETAILS ===")
print(network.get("SupplierA", "SupplierA not found"))
print("\n=== SUPPLIER NETWORK REPORT ===")
for supplier, details in sorted(network.items()):
    print(f"{supplier} | Orders: {details['order_count']:,} | "
          f"Categories: {len(details['categories'])} | "
          f"Total spend: {details['total_spend']:,.2f}")
most_spend_supplier, most_spend_info = max(network.items(), key=lambda pair: pair[1]["total_spend"])
print(f"\nHighest spend: {most_spend_supplier} ({most_spend_info['total_spend']:,.2f})")

=== SUPPLIER A DETAILS ===
{'categories': {'Motors', 'Control Systems'}, 'order_count': 769, 'total_spend': 70637594.46094}

=== SUPPLIER NETWORK REPORT ===
SupplierA | Orders: 769 | Categories: 2 | Total spend: 70,637,594.46
SupplierB | Orders: 659 | Categories: 3 | Total spend: 43,099,108.00
SupplierC | Orders: 492 | Categories: 3 | Total spend: 23,085,480.95
SupplierD | Orders: 433 | Categories: 2 | Total spend: 28,449,003.78
SupplierE | Orders: 647 | Categories: 2 | Total spend: 4,766,913.09

Highest spend: SupplierA (70,637,594.46)


## Extra challenge 1: High-value orders
I counted orders with a recorded cost strictly above 100,000. The percentage denominator was **all dataset orders**.

In [8]:
high_value_orders = [row for row in rows
                     if row["total_cost"].strip() and float(row["total_cost"]) > 100000]
print(f"Orders above 100,000: {len(high_value_orders):,}")
print(f"Share of all orders: {len(high_value_orders) / len(rows) * 100:.2f}%")

Orders above 100,000: 478
Share of all orders: 15.93%


## Extra challenge 2: Supplier with the most categories
I reported all suppliers tied for the highest number of categories, where applicable.

In [9]:
max_categories = max(len(info["categories"]) for info in network.values())
leaders = sorted(supplier for supplier, info in network.items()
                 if len(info["categories"]) == max_categories)
print(f"Most categories ({max_categories}): {', '.join(leaders)}")

Most categories (3): SupplierB, SupplierC


## Extra challenge 3: Average delivered order value
I grouped recorded costs by the original delivery status, and calculated the average for the exact `Delivered` label.

In [10]:
costs_by_status = {}
for row in rows:
    status = row["delivery_status"]
    if row["total_cost"].strip():
        costs_by_status.setdefault(status, []).append(float(row["total_cost"]))
for status, costs in sorted(costs_by_status.items()):
    print(f"{status!r}: {len(costs):,} costed orders")
delivered_costs = costs_by_status.get("Delivered", [])
if delivered_costs:
    print(f"Average 'Delivered' order value: {sum(delivered_costs)/len(delivered_costs):,.2f}")
else:
    print("No exact 'Delivered' records with recorded costs")

'Cancelled': 64 costed orders
'Delayed': 531 costed orders
'Delievered': 22 costed orders
'Delivered': 2,340 costed orders
'In-Transit': 22 costed orders
'Pendng': 21 costed orders
Average 'Delivered' order value: 57,473.02


## Supply Chain Network Summary
The supply chain dataset contains 3,000 orders across 5 real suppliers and 8 product categories after standardizing supplier names and trimming category whitespace. 4 categories are single-source: Connectors (SupplierE); Control Systems (SupplierA); Fasteners (SupplierE); Hydraulics (SupplierD). These categories are exposed to disruption because losing their sole supplier could interrupt supply. SupplierA receives the highest recorded spend at 70,637,594.46 (currency not specified in the dataset).

Source: Week 2 supply_chain_dataset.csv (3,000 orders).

GitHub repository: https://github.com/Johndgenius23/TDI-Python-2.0
